In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [3]:
len(words)

32033

In [4]:
# build the vocabulary of charachers and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [7]:
# build the dataset

block_size = 3 # context length: how many characters do we take to predict the next one?
X, Y = [], []
for w in words[:5]:

    print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context), '--->', itos[ix])
        context = context[1:] + [ix] # crop and append
X = torch.tensor(X)
Y = torch.tensor(Y)

emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [8]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [11]:
# build the "lookup table" 'C'
# 27 characters, embedded in a lower dimensional space
C = torch.randn((27, 2))

In [12]:
C

tensor([[-0.0354, -1.4306],
        [-0.9840, -0.8871],
        [ 0.5669,  1.0111],
        [-1.1782, -0.6637],
        [-0.4649, -0.0326],
        [ 0.8625, -0.3241],
        [ 0.8282, -0.4651],
        [-0.4580,  1.2333],
        [-0.6960,  0.9445],
        [ 1.0797, -0.1513],
        [ 1.2807,  0.6004],
        [ 2.4414,  0.1123],
        [ 1.5759, -0.8779],
        [ 1.1060, -1.0243],
        [ 0.5325,  0.5875],
        [-1.7314,  1.7486],
        [ 0.1305,  0.1327],
        [-0.2705,  0.1210],
        [ 0.3005,  0.1520],
        [-0.1576, -1.8266],
        [ 0.1181, -0.8854],
        [ 0.1609,  0.6230],
        [ 0.7933,  0.6770],
        [ 0.0746, -0.6078],
        [ 0.3140,  1.8194],
        [ 0.4549, -2.3318],
        [ 1.1067, -0.7212]])

In [13]:
# 2 ways to think about indexing into C
# initial:

In [14]:
C[5]

tensor([ 0.8625, -0.3241])

In [15]:
# second:
F.one_hot(torch.tensor(5), num_classes=27).float() @ C

tensor([ 0.8625, -0.3241])

In [16]:
# can think of the indexing as a first layer of the neural net
# there is no non-linearity (tanh), weight matrix is C
# encoding integers into one_hot and feeding in
# this 'first layer' embeds them

In [17]:
# different ways of indexing
C[5]

tensor([ 0.8625, -0.3241])

In [18]:
C[[5, 6, 7]]

tensor([[ 0.8625, -0.3241],
        [ 0.8282, -0.4651],
        [-0.4580,  1.2333]])

In [20]:
C[torch.tensor([5, 6, 7, 7, 7])]

tensor([[ 0.8625, -0.3241],
        [ 0.8282, -0.4651],
        [-0.4580,  1.2333],
        [-0.4580,  1.2333],
        [-0.4580,  1.2333]])

In [21]:
# index w/ multidimensional vectors
C[X]

tensor([[[-0.0354, -1.4306],
         [-0.0354, -1.4306],
         [-0.0354, -1.4306]],

        [[-0.0354, -1.4306],
         [-0.0354, -1.4306],
         [ 0.8625, -0.3241]],

        [[-0.0354, -1.4306],
         [ 0.8625, -0.3241],
         [ 1.1060, -1.0243]],

        [[ 0.8625, -0.3241],
         [ 1.1060, -1.0243],
         [ 1.1060, -1.0243]],

        [[ 1.1060, -1.0243],
         [ 1.1060, -1.0243],
         [-0.9840, -0.8871]],

        [[-0.0354, -1.4306],
         [-0.0354, -1.4306],
         [-0.0354, -1.4306]],

        [[-0.0354, -1.4306],
         [-0.0354, -1.4306],
         [-1.7314,  1.7486]],

        [[-0.0354, -1.4306],
         [-1.7314,  1.7486],
         [ 1.5759, -0.8779]],

        [[-1.7314,  1.7486],
         [ 1.5759, -0.8779],
         [ 1.0797, -0.1513]],

        [[ 1.5759, -0.8779],
         [ 1.0797, -0.1513],
         [ 0.7933,  0.6770]],

        [[ 1.0797, -0.1513],
         [ 0.7933,  0.6770],
         [ 1.0797, -0.1513]],

        [[ 0.7933,  0

In [22]:
C[X].shape

torch.Size([32, 3, 2])

In [23]:
X[13,2]

tensor(1)

In [24]:
C[X][13,2]

tensor([-0.9840, -0.8871])

In [25]:
C[1]

tensor([-0.9840, -0.8871])

In [26]:
# in summary pytorch indexing is awesom!

In [28]:
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

In [31]:
# construct hidden layer
# weights initialized randomly, 6 dimensions for (3 * 2) from input shape above
# num neurons is up to us, 100 for example
W1 = torch.randn((6, 100))
# biases also initialized randomly
b1 = torch.randn(100)

In [32]:
# need to 'concatenate' inputs in [32, 3, 2] 
# so we can mult by [6, 100]
# as in emb @ W1 + b1

In [33]:
# torch.cat does just that
# this will not generalize if we change block size
torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], 1).shape

torch.Size([32, 6])

In [36]:
# torch.unbind does what we need
# pass in the embedding and specify a dimension
# returns a tuple of slices along a given dimension, already without it
torch.unbind(emb, 1)

(tensor([[-0.0354, -1.4306],
         [-0.0354, -1.4306],
         [-0.0354, -1.4306],
         [ 0.8625, -0.3241],
         [ 1.1060, -1.0243],
         [-0.0354, -1.4306],
         [-0.0354, -1.4306],
         [-0.0354, -1.4306],
         [-1.7314,  1.7486],
         [ 1.5759, -0.8779],
         [ 1.0797, -0.1513],
         [ 0.7933,  0.6770],
         [-0.0354, -1.4306],
         [-0.0354, -1.4306],
         [-0.0354, -1.4306],
         [-0.9840, -0.8871],
         [-0.0354, -1.4306],
         [-0.0354, -1.4306],
         [-0.0354, -1.4306],
         [ 1.0797, -0.1513],
         [-0.1576, -1.8266],
         [-0.9840, -0.8871],
         [ 0.5669,  1.0111],
         [ 0.8625, -0.3241],
         [ 1.5759, -0.8779],
         [-0.0354, -1.4306],
         [-0.0354, -1.4306],
         [-0.0354, -1.4306],
         [-0.1576, -1.8266],
         [-1.7314,  1.7486],
         [ 0.1305,  0.1327],
         [-0.6960,  0.9445]]),
 tensor([[-0.0354, -1.4306],
         [-0.0354, -1.4306],
         [ 0

In [37]:
torch.cat(torch.unbind(emb, 1), 1).shape

torch.Size([32, 6])

In [38]:
# there is still another better way!

In [40]:
# ASIDE>>>
a = torch.arange(18)
a

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17])

In [41]:
a.shape

torch.Size([18])

In [42]:
a.view(2, 9)

tensor([[ 0,  1,  2,  3,  4,  5,  6,  7,  8],
        [ 9, 10, 11, 12, 13, 14, 15, 16, 17]])

In [43]:
a.view(9, 2)

tensor([[ 0,  1],
        [ 2,  3],
        [ 4,  5],
        [ 6,  7],
        [ 8,  9],
        [10, 11],
        [12, 13],
        [14, 15],
        [16, 17]])

In [44]:
a.view(3,3,2)

tensor([[[ 0,  1],
         [ 2,  3],
         [ 4,  5]],

        [[ 6,  7],
         [ 8,  9],
         [10, 11]],

        [[12, 13],
         [14, 15],
         [16, 17]]])

In [47]:
# tensor always represented in computer memory as 1-dimensional vector
a.storage()

 0
 1
 2
 3
 4
 5
 6
 7
 8
 9
 10
 11
 12
 13
 14
 15
 16
 17
[torch.storage.TypedStorage(dtype=torch.int64, device=cpu) of size 18]

In [48]:
emb.view(32, 6)

tensor([[-0.0354, -1.4306, -0.0354, -1.4306, -0.0354, -1.4306],
        [-0.0354, -1.4306, -0.0354, -1.4306,  0.8625, -0.3241],
        [-0.0354, -1.4306,  0.8625, -0.3241,  1.1060, -1.0243],
        [ 0.8625, -0.3241,  1.1060, -1.0243,  1.1060, -1.0243],
        [ 1.1060, -1.0243,  1.1060, -1.0243, -0.9840, -0.8871],
        [-0.0354, -1.4306, -0.0354, -1.4306, -0.0354, -1.4306],
        [-0.0354, -1.4306, -0.0354, -1.4306, -1.7314,  1.7486],
        [-0.0354, -1.4306, -1.7314,  1.7486,  1.5759, -0.8779],
        [-1.7314,  1.7486,  1.5759, -0.8779,  1.0797, -0.1513],
        [ 1.5759, -0.8779,  1.0797, -0.1513,  0.7933,  0.6770],
        [ 1.0797, -0.1513,  0.7933,  0.6770,  1.0797, -0.1513],
        [ 0.7933,  0.6770,  1.0797, -0.1513, -0.9840, -0.8871],
        [-0.0354, -1.4306, -0.0354, -1.4306, -0.0354, -1.4306],
        [-0.0354, -1.4306, -0.0354, -1.4306, -0.9840, -0.8871],
        [-0.0354, -1.4306, -0.9840, -0.8871,  0.7933,  0.6770],
        [-0.9840, -0.8871,  0.7933,  0.6

In [49]:
emb.view(32, 6) == torch.cat(torch.unbind(emb, 1), 1)

tensor([[True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, T

In [56]:
# Finally
# h = emb.view(emb.shape[0], 6) @ W1 + b1
# shape can be inferred if you use '-1' as a placeholder
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)

In [57]:
h

tensor([[ 0.9943, -0.0952, -0.9997,  ..., -0.5317,  0.6344,  0.6283],
        [ 0.9960,  0.9307, -0.9889,  ...,  0.9768, -0.9733,  0.9949],
        [ 0.9801,  0.6950, -0.8176,  ...,  0.2666, -0.8317, -0.4212],
        ...,
        [-0.9091,  1.0000,  0.7642,  ...,  1.0000,  0.9965, -0.7284],
        [-0.9936, -0.8911,  0.6997,  ...,  0.9818,  0.0435, -0.9787],
        [ 0.8793,  0.9989, -0.8140,  ..., -0.7078,  0.9999, -0.9998]])

In [52]:
h.shape

torch.Size([32, 100])